In [1]:
import json
import random
import time
import sys
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

load_dotenv(Path('../../.env'))

# add app to path so we can import ingest
sys.path.append('../app')
import ingest

In [2]:
index, vindex, embedding_model = ingest.index_data(
    'DataTalksClub', 
    'data-engineering-zoomcamp'
)

# we need the raw chunks for question generation
zoomcamp_docs = ingest.read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')
zoomcamp_chunks = ingest.build_sections(zoomcamp_docs)
print(f"Total chunks: {len(zoomcamp_chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks: 620


In [5]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering zoomcamp.

Based on the provided content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt + "\n\nReturn ONLY a JSON object like: {\"questions\": [\"question 1\", \"question 2\"]}. No tags, no markdown.",
    model=GroqModel('llama-3.3-70b-versatile'),
)


In [7]:
import re

async def generate_questions(chunks, n=20):
    sample = random.sample(chunks, n)
    prompt_docs = [d['section'] for d in sample]
    prompt = json.dumps(prompt_docs)
    
    result = await question_generator.run(prompt)
    raw = result.output
    
    # strip function tags
    raw = re.sub(r'<function[^>]*>', '', raw)
    raw = re.sub(r'</function>', '', raw).strip()
    
    # extract JSON
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        data = json.loads(json_match.group())
        return data.get('questions', [])
    
    raise ValueError(f"Could not parse: {raw[:200]}")

questions = await generate_questions(zoomcamp_chunks, n=20)
print(f"Generated {len(questions)} questions")
for q in questions:
    print(f"- {q}")

Generated 10 questions
- What is the purpose of the 'zone' table in the SQL query provided for Question 5?
- How do I submit ad-hoc queries against my connections using 'bruin query'?
- What is the purpose of Prefect Secret blocks and how do I create one?
- What is the difference between a pass-through Flink job and a regular Flink job?
- What are the two main things you want in your data marts, according to the provided text?
- How do I initialize a new project using 'bruin init'?
- What is the purpose of the 'resolve_schema_for' macro in dbt and how does it work?
- How do I troubleshoot issues in the Data Engineering Zoomcamp course?
- What is the best way to ask questions in the course's Slack channel?
- How do I clean up local files after completing an exercise or project?


In [8]:
with open('questions.json', 'w') as f:
    json.dump(questions, f, indent=2)

print(f"Saved {len(questions)} questions to questions.json")

Saved 10 questions to questions.json
